## Compile Virchow to TorchScript

Source: https://huggingface.co/paige-ai/Virchow

We serialize Virchow so that we can serve it via Triton. We bake in some steps as well assuming a deployed model pipeline in kserve:
* Normalization: Normalization in the kserve transformer can be CPU heavy for thousands of tiles. Additionally, sending fp32 tensors between kserve transformer and predictor (this serialized virchow) is more expensive than sending uint8. So we add it to the start of the model instead, which has the advantage of GPU acceleration.
* Embedding: Virchow has a specific way of deriving tile embeddings after the model output. We bake in the concatenation of the classification token and the averaged patch tokens.

In [1]:
import timm
import torch
import torch.nn as nn
from timm.layers import SwiGLUPacked


class WrappedVirchow(nn.Module):
    """
    Expects 224x224 px (20x @ 0.5MPP) RGB input images as uint8 Tensor of shape (B, C, H, W),
    where B is batch size, C = 3 channels (ordered RGB), and H = W = 224.
    """

    def __init__(self):
        super().__init__()
        self.virchow = timm.create_model(
            "hf-hub:paige-ai/Virchow",
            pretrained=True,
            mlp_layer=SwiGLUPacked,
            act_layer=torch.nn.SiLU,
        )

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        # torch.jit.trace doesn't preserve assertions so can't check shape of x
        x = (x.float() / 255.0 - self.mean) / self.std # (B, 3, 224, 224)
        x = self.virchow(x) # (B, 257, 1280)

        cls_emb = x[:, 0] # (B, 1280)
        patch_embs = x[:, 1:] # (B, 256, 1280)

        # concatenate class token and average pool of patch tokens
        x = torch.cat([cls_emb, patch_embs.mean(1)], dim=-1) # (B, 2560)

        return x

### Serialize to TorchScript

In [ ]:
import os
os.makedirs("../model-repo/virchow/1", exist_ok=True)

In [ ]:
model = WrappedVirchow()
model.eval()

x = torch.randint(0, 256, (1, 3, 224, 224), dtype=torch.uint8)
traced_script_module = torch.jit.trace(model, x)
traced_script_module.save("../model-repo/virchow/1/model.pt")

/home/songs1/miniforge3/envs/wsi/lib/python3.12/site-packages/torch/__init__.py:2278: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not condition:
/home/songs1/miniforge3/envs/wsi/lib/python3.12/site-packages/timm/layers/pos_embed.py:31: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_new_tokens == num_pos_tokens and new_size[0] == new_size[1]:


### Ensure Self-Consistency

In [ ]:
# 1. Create a fresh test input (matching the uint8 format you traced with)
test_input = torch.randint(0, 256, (1, 3, 224, 224), dtype=torch.uint8)

# 2. Load original model
base_model = WrappedVirchow()
base_model.eval()

# 3. Load the saved TorchScript model
trace_model = torch.jit.load("../model-repo/virchow/1/model.pt")
trace_model.eval()

# 4. Run both models under no_grad to prevent memory leaks and ensure pure inference
with torch.no_grad():
    original_output = base_model(test_input)
    traced_output = trace_model(test_input)

# 5. Compare the results
# atol=1e-5 sets the absolute tolerance for floating point math differences
is_equivalent = torch.allclose(original_output, traced_output, atol=1e-5)
max_diff = (original_output - traced_output).abs().max().item()

print(f"Models equivalent: {is_equivalent}")
print(f"Maximum absolute difference: {max_diff:.8f}")

if not is_equivalent:
    print("Warning: Models diverge beyond the allowed tolerance!")

Models equivalent: True
Maximum absolute difference: 0.00000000
